# Gluten/Velox — 03: Off-Heap Memory Management and Profiling

## What you will learn
Off-heap memory is one of the most misunderstood aspects of Spark tuning.
It is critical for understanding why Gluten/Velox performs well and how
to configure memory correctly for production workloads.

In this notebook you will:
1. Understand the difference between on-heap and off-heap memory in Spark
2. Learn how Gluten/Velox uses native (off-heap) memory
3. Profile memory usage in both vanilla and Gluten mode
4. Configure off-heap settings correctly
5. Diagnose and fix OOM errors related to off-heap
6. Monitor memory via Spark UI and REST API

## Memory in Spark: Three Separate Pools

```
┌──────────────────────────────────────────────────────────────┐
│  EXECUTOR PROCESS                                            │
│                                                              │
│  1. JVM ON-HEAP (spark.executor.memory = 2g)                 │
│     ├── Reserved: 300 MB  (Spark internals)                  │
│     ├── User:     ~700 MB (your code, UDFs, collections)     │
│     └── Pool:     ~1 GB   (Storage cache + Execution)        │
│           ├── Storage:   ~500 MB (cached DataFrames)         │
│           └── Execution: ~500 MB (shuffle, sort, join)       │
│                                                              │
│  2. OFF-HEAP (spark.memory.offHeap.size)                     │
│     └── Tungsten sort/shuffle buffers (NOT GC'd)             │
│                                                              │
│  3. OVERHEAD (spark.executor.memoryOverhead = 512m)          │
│     ├── JVM metadata (class loading, JIT)                    │
│     ├── Python worker process (PySpark)                      │
│     └── Native libraries (NIO buffers, Gluten/Velox)         │
│                                                              │
│  GLUTEN/VELOX ADDITIONALLY:                                  │
│  4. VELOX NATIVE MEMORY POOL                                 │
│     └── All columnar data lives here — completely off JVM    │
└──────────────────────────────────────────────────────────────┘
```

**Why Gluten reduces GC pressure:**
In vanilla Spark, every Row, InternalRow, and UnsafeRow lives on the JVM heap.
With Gluten, all columnar batch data lives in Velox's native memory pool —
completely outside the JVM heap and GC.
The JVM only sees lightweight wrapper objects, not the actual data.


In [ ]:
import os, time, random, datetime, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'
GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'

spark = (
    SparkSession.builder
    .appName('off-heap-memory')
    .master(MASTER)
    .config('spark.executor.memory',            '2g')
    .config('spark.executor.memoryOverhead',    '512m')
    .config('spark.driver.memory',              '1g')
    .config('spark.executor.cores',             '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
mode = "GLUTEN/VELOX" if GLUTEN_ENABLED else "VANILLA JVM"
print(f"Spark {spark.version} | Mode: {mode}")

## Step 1 — Inspect Current Memory Configuration

Let's see exactly how memory is split across all three pools.


In [ ]:
def get_conf(key, default="(default)"):
    try:    return spark.conf.get(key)
    except: return default

print("=" * 65)
print("COMPLETE MEMORY CONFIGURATION")
print("=" * 65)

sections = {
    "On-Heap (JVM)": [
        ("spark.executor.memory",           "2g",    "JVM heap size"),
        ("spark.memory.fraction",           "0.6",   "fraction for Spark pool"),
        ("spark.memory.storageFraction",    "0.5",   "storage fraction of pool"),
    ],
    "Off-Heap (Tungsten)": [
        ("spark.memory.offHeap.enabled",    "false", "enable off-heap pool"),
        ("spark.memory.offHeap.size",       "0",     "off-heap pool size"),
    ],
    "Overhead (Container)": [
        ("spark.executor.memoryOverhead",   "512m",  "container overhead"),
        ("spark.executor.memoryOverheadFactor","0.1","fraction of executor mem"),
    ],
    "Gluten/Velox Native": [
        ("spark.gluten.memory.offHeap.size.in.bytes", "n/a", "Velox memory pool"),
        ("spark.gluten.columnar.offHeap.size",        "n/a", "Gluten off-heap size"),
    ],
}

for section, settings in sections.items():
    print(f"\n  {section}:")
    for key, default, desc in settings:
        val = get_conf(key, default)
        print(f"    {key.split('.')[-1]:<40} {val:<12} ({desc})")

# Calculate total container memory
executor_mb  = 2048
overhead_mb  = 512
total_mb     = executor_mb + overhead_mb
print(f"\n  Total container memory per executor: {total_mb} MB")
print(f"  (spark.executor.memory + spark.executor.memoryOverhead)")

if GLUTEN_ENABLED:
    print(f"\n  GLUTEN MODE: Velox native memory pool is active.")
    print(f"  All columnar data lives in native memory (off JVM heap).")
    print(f"  JVM heap usage is significantly lower than vanilla mode.")

## Step 2 — On-Heap vs Off-Heap: A Concrete Example

Let's observe JVM heap usage while running the same query
in vanilla mode vs Gluten mode.

In vanilla mode the JVM heap fills up during query execution.
In Gluten mode the JVM heap stays relatively free — data is in Velox's pool.


In [ ]:
import urllib.request, json as pyjson, threading

def get_executor_memory():
    """Poll executor memory metrics via Spark REST API."""
    try:
        url = "http://spark-master:4040/api/v1/applications"
        with urllib.request.urlopen(url, timeout=3) as r:
            apps = pyjson.loads(r.read())
        if not apps:
            return None
        app_id = apps[0]["id"]
        url2 = f"http://spark-master:4040/api/v1/applications/{app_id}/executors"
        with urllib.request.urlopen(url2, timeout=3) as r:
            executors = pyjson.loads(r.read())
        metrics = []
        for ex in executors:
            if ex["id"] == "driver":
                continue
            pm = ex.get("peakMemoryMetrics", {})
            metrics.append({
                "id":              ex["id"],
                "jvm_heap_mb":     pm.get("JVMHeapMemory",    0) / 1024 / 1024,
                "jvm_offheap_mb":  pm.get("JVMOffHeapMemory", 0) / 1024 / 1024,
                "on_heap_exec_mb": pm.get("OnHeapExecutionMemory", 0) / 1024 / 1024,
                "off_heap_exec_mb":pm.get("OffHeapExecutionMemory",0) / 1024 / 1024,
                "storage_mb":      pm.get("OnHeapStorageMemory", 0) / 1024 / 1024,
                "process_tree_mb": pm.get("ProcessTreeJVMVMemory",0) / 1024 / 1024,
            })
        return metrics
    except Exception as e:
        return None

# Baseline memory before any query
print("Memory BEFORE query:")
before = get_executor_memory()
if before:
    for ex in before:
        print(f"  Executor {ex['id']}: JVM heap={ex['jvm_heap_mb']:.0f} MB  "
              f"off-heap={ex['jvm_offheap_mb']:.0f} MB  "
              f"exec={ex['on_heap_exec_mb']:.0f} MB")
else:
    print("  (Spark UI not available — run a query first to start the app)")

In [ ]:
# Run a memory-intensive query and observe heap
random.seed(42)
N = 2_000_000

heavy_df = spark.range(N).select(
    F.col("id"),
    (F.col("id") % 100_000).alias("key"),
    F.rand(42).alias("v1"),
    F.rand(43).alias("v2"),
    F.rand(44).alias("v3"),
    F.concat(F.lit("item_"), F.col("id").cast("string")).alias("label"),
)

print(f"Running heavy aggregation ({N:,} rows)...")
t0 = time.time()
result = (
    heavy_df
    .groupBy("key")
    .agg(F.sum("v1"), F.avg("v2"), F.max("v3"), F.count("*"))
    .agg(F.sum("sum(v1)"), F.count("*"))
    .collect()
)
elapsed = time.time() - t0
print(f"Completed in {elapsed:.2f}s | Result: {result[0]}")
print()

# Memory AFTER query
print(f"Memory AFTER query ({mode} mode):")
after = get_executor_memory()
if after:
    for ex in after:
        print(f"  Executor {ex['id']}:")
        print(f"    JVM heap used   : {ex['jvm_heap_mb']:>7.0f} MB")
        print(f"    JVM off-heap    : {ex['jvm_offheap_mb']:>7.0f} MB")
        print(f"    On-heap exec    : {ex['on_heap_exec_mb']:>7.0f} MB")
        print(f"    Off-heap exec   : {ex['off_heap_exec_mb']:>7.0f} MB")
        print(f"    Storage (cache) : {ex['storage_mb']:>7.0f} MB")
    print()
    if GLUTEN_ENABLED:
        print("GLUTEN MODE: JVM heap should be significantly lower than vanilla.")
        print("Velox data lives in native memory (reported under process RSS,")
        print("not under JVM heap metrics).")
    else:
        print("VANILLA MODE: JVM heap rises during aggregation as row objects")
        print("are created and GC'd. Compare with Gluten mode run.")

## Step 3 — GC Pressure: Vanilla vs Gluten

JVM Garbage Collection is one of the biggest performance killers in Spark.
When the GC runs, all JVM threads pause — your executors do no useful work.

In vanilla mode: every shuffle, sort, and aggregation creates JVM objects → GC.
In Gluten mode: columnar data is in Velox native memory → no GC for data processing.


In [ ]:
# Run a workload and measure GC time
import urllib.request, json as pyjson

def get_gc_metrics():
    try:
        url = "http://spark-master:4040/api/v1/applications"
        with urllib.request.urlopen(url, timeout=3) as r:
            apps = pyjson.loads(r.read())
        if not apps: return None
        app_id = apps[0]["id"]
        url2 = f"http://spark-master:4040/api/v1/applications/{app_id}/executors"
        with urllib.request.urlopen(url2, timeout=3) as r:
            executors = pyjson.loads(r.read())
        return {ex["id"]: {
            "gc_time_ms":    ex.get("totalGCTime", 0),
            "task_time_ms":  ex.get("totalDuration", 0),
            "jvm_gc_ratio":  ex.get("totalGCTime", 0) / max(ex.get("totalDuration", 1), 1) * 100
        } for ex in executors if ex["id"] != "driver"}
    except:
        return None

gc_before = get_gc_metrics()

# Run multiple heavy queries to accumulate GC
for i in range(3):
    spark.range(1_000_000).select(
        (F.col("id") % 10_000).alias("k"),
        F.rand(i).alias("v")
    ).groupBy("k").agg(F.sum("v"), F.avg("v"), F.count("*")).collect()

gc_after = get_gc_metrics()

if gc_before and gc_after:
    print(f"GC Metrics after 3 heavy aggregations ({mode} mode):")
    print(f"  {'Executor':<12} {'GC Time (ms)':>14} {'Task Time (ms)':>16} {'GC %':>8}")
    print("  " + "-" * 55)
    for ex_id, after in gc_after.items():
        before_gc = gc_before.get(ex_id, {}).get("gc_time_ms", 0)
        delta_gc  = after["gc_time_ms"] - before_gc
        ratio     = after["jvm_gc_ratio"]
        print(f"  {ex_id:<12} {delta_gc:>12,} ms {after['task_time_ms']:>14,} ms {ratio:>7.1f}%")
    print()
    if GLUTEN_ENABLED:
        print("GLUTEN: GC time should be <5% of task time.")
        print("Velox processes data in native memory — JVM GC only runs for wrapper objects.")
    else:
        print("VANILLA: GC time may reach 10-20% of task time under heavy load.")
        print("Each InternalRow/UnsafeRow allocation triggers GC cycles.")
else:
    print("GC metrics not available via REST API.")
    print("Check http://localhost:4040/executors for GC time manually.")

## Step 4 — Enabling Tungsten Off-Heap

Tungsten off-heap moves sort and shuffle buffers outside the JVM heap.
This reduces GC pressure for sort-heavy workloads even in vanilla mode.

**Note:** Gluten/Velox uses its own native memory pool which is separate
from Tungsten off-heap. Enabling Tungsten off-heap with Gluten gives
**two** separate off-heap pools.


In [ ]:
print("Current Tungsten off-heap settings:")
print(f"  spark.memory.offHeap.enabled : {get_conf('spark.memory.offHeap.enabled', 'false')}")
print(f"  spark.memory.offHeap.size    : {get_conf('spark.memory.offHeap.size', '0')}")
print()
print("To enable Tungsten off-heap, add to SparkSession builder:")
print()
print("  .config('spark.memory.offHeap.enabled', 'true')")
print("  .config('spark.memory.offHeap.size',    '1g')")
print()
print("This moves sort/shuffle buffers out of JVM heap → less GC.")
print()
print("Memory layout with off-heap enabled:")
print()
print("  JVM heap (2g):")
print("    ├── Reserved: 300 MB")
print("    ├── User:     ~700 MB")
print("    └── Spark Pool: ~1 GB  ← SMALLER because off-heap takes over execution")
print()
print("  Tungsten off-heap (1g):")
print("    └── Sort buffers, shuffle write buffers")
print()
print("  Container overhead (512m):")
print("    └── JVM meta, Python worker, NIO")
print()
if GLUTEN_ENABLED:
    print("  Velox native pool (configured separately):")
    print("    └── ALL columnar data, hash tables, sort results")
    print()
    print("With Gluten: Tungsten off-heap is less important because Velox")
    print("handles its own memory. Focus on executor.memoryOverhead instead.")

## Step 5 — Velox Memory Configuration (Gluten Mode)

When running with Gluten, Velox has its own memory management system.
Understanding and tuning it is essential for production deployments.


In [ ]:
if GLUTEN_ENABLED:
    print("Gluten/Velox memory settings:")
    velox_settings = [
        ("spark.gluten.memory.offHeap.size.in.bytes", "auto"),
        ("spark.gluten.columnar.offHeap.size",        "auto"),
        ("spark.gluten.memory.reserved.size.in.bytes","auto"),
        ("spark.executor.memoryOverhead",             get_conf("spark.executor.memoryOverhead","512m")),
    ]
    for key, default in velox_settings:
        val = get_conf(key, default)
        print(f"  {key.split('.')[-1]:<45} {val}")

    print()
    print("Velox memory architecture:")
    print()
    print("  Velox uses a MemoryManager with a configurable memory pool.")
    print("  By default it takes spark.executor.memoryOverhead as budget.")
    print()
    print("  Recommended production config:")
    print("    spark.executor.memory              = 4g  (JVM heap)")
    print("    spark.executor.memoryOverhead      = 2g  (for Velox native pool)")
    print("    spark.gluten.memory.offHeap.size   = 2g  (Velox pool size)")
    print()
    print("  Rule of thumb:")
    print("    JVM heap   = 40% of total executor memory")
    print("    Velox pool = 50% of total executor memory")
    print("    Overhead   = 10% for JVM metadata, Python worker")
else:
    print("Running in VANILLA mode.")
    print("Switch to Gluten mode (make up-gluten) to see Velox memory settings.")
    print()
    print("In Gluten mode, Velox uses spark.executor.memoryOverhead as its")
    print("native memory budget. The JVM heap is used only for Spark metadata.")

## Step 6 — Memory Profiling: Cache vs Execution Trade-off

The Spark memory pool is shared between Storage (cache) and Execution (shuffle/sort/join).
When you cache a large DataFrame, it competes with execution memory.
This is the **unified memory** model.

Let's observe this trade-off in practice.


In [ ]:
# Create a large dataset to cache
large_for_cache = spark.range(2_000_000).select(
    F.col("id"),
    (F.col("id") % 1_000).alias("group"),
    F.rand(42).alias("val1"),
    F.rand(43).alias("val2"),
    F.concat(F.lit("label_"), (F.col("id") % 10_000).cast("string")).alias("label")
)

print("Scenario 1: Run heavy query WITHOUT caching baseline")
t0 = time.time()
large_for_cache.groupBy("group").agg(
    F.sum("val1"), F.avg("val2"), F.count("*")
).count()
t_no_cache = time.time() - t0
print(f"  No cache: {t_no_cache:.2f}s")

print()
print("Scenario 2: Cache the DataFrame, then run query")
large_for_cache.cache()
large_for_cache.count()  # materialize cache

storage_after = get_executor_memory()
if storage_after:
    cached_mb = sum(ex["storage_mb"] for ex in storage_after)
    print(f"  Cache used: {cached_mb:.0f} MB across all executors")

t0 = time.time()
large_for_cache.groupBy("group").agg(
    F.sum("val1"), F.avg("val2"), F.count("*")
).count()
t_cached = time.time() - t0
print(f"  With cache: {t_cached:.2f}s  (speedup: {t_no_cache/t_cached:.1f}x)")

print()
print("The cache uses Storage memory. If a concurrent query needs Execution")
print("memory, Spark can evict cached partitions — they get recomputed on next access.")
print("This is the unified memory model: cache and execution share the pool.")

large_for_cache.unpersist()
print()
print("Cache released.")

## Step 7 — Diagnosing OOM Errors

Different OOM errors have different causes and solutions.


In [ ]:
print("OOM Error Diagnosis Guide")
print("=" * 65)
print()

issues = [
    {
        "error":   "java.lang.OutOfMemoryError: Java heap space",
        "cause":   "JVM heap exhausted — too much data in on-heap memory",
        "fixes":   [
            "Increase spark.executor.memory (e.g., 2g → 4g)",
            "Reduce data: filter earlier, select fewer columns",
            "Use MEMORY_AND_DISK cache instead of MEMORY_ONLY",
            "Increase spark.sql.shuffle.partitions (200 → 500)",
        ]
    },
    {
        "error":   "java.lang.OutOfMemoryError: GC overhead limit exceeded",
        "cause":   "GC running >98% of time — heap full of short-lived objects",
        "fixes":   [
            "Enable off-heap: spark.memory.offHeap.enabled=true",
            "Use Kryo serializer: spark.serializer=KryoSerializer",
            "Switch to Gluten/Velox (eliminates GC on data objects)",
            "Reduce UDF usage (UDFs create many JVM objects)",
        ]
    },
    {
        "error":   "ExecutorLostFailure: Container killed by YARN for exceeding memory",
        "cause":   "Container memory (heap + overhead) exceeded limit",
        "fixes":   [
            "Increase spark.executor.memoryOverhead (384m → 1g)",
            "Reduce spark.executor.memory to leave room for overhead",
            "In Gluten mode: increase overhead (Velox pool is in overhead budget)",
        ]
    },
    {
        "error":   "org.apache.spark.shuffle.FetchFailedException",
        "cause":   "Shuffle data lost — executor OOM'd mid-shuffle",
        "fixes":   [
            "Increase shuffle partitions to reduce per-partition size",
            "Increase executor memory",
            "Enable off-heap for shuffle buffers",
        ]
    },
]

for issue in issues:
    print(f"ERROR: {issue['error']}")
    print(f"  Cause: {issue['cause']}")
    print(f"  Fixes:")
    for fix in issue["fixes"]:
        print(f"    • {fix}")
    print()

## Summary: Off-Heap and Memory Configuration Guide

### Memory configuration for different workloads

**Vanilla Spark — cache-heavy workload:**
```python
.config('spark.executor.memory',         '4g')    # heap
.config('spark.executor.memoryOverhead', '512m')  # container overhead
.config('spark.memory.fraction',         '0.6')   # 60% for Spark pool
.config('spark.memory.storageFraction',  '0.7')   # more for cache
```

**Vanilla Spark — shuffle-heavy workload:**
```python
.config('spark.executor.memory',          '4g')
.config('spark.memory.offHeap.enabled',   'true')
.config('spark.memory.offHeap.size',      '2g')   # sort/shuffle buffers off-heap
.config('spark.executor.memoryOverhead',  '512m')
.config('spark.memory.storageFraction',   '0.3')  # less cache, more execution
```

**Gluten/Velox mode:**
```python
.config('spark.executor.memory',          '3g')   # JVM heap (less needed)
.config('spark.executor.memoryOverhead',  '3g')   # Velox native pool lives here
.config('spark.memory.fraction',          '0.6')
.config('spark.memory.storageFraction',   '0.5')
```

### Key rules
1. **Total container memory = executor.memory + executor.memoryOverhead**
2. **Gluten needs large overhead** — Velox data is in overhead budget
3. **Enable off-heap for GC problems** in vanilla mode
4. **Monitor GC %** via Spark UI — should be < 5% of task time
5. **Storage eviction is normal** — Spark evicts cache when execution needs memory

### Monitoring commands
```python
# GC time per executor
spark.sparkContext._jsc.sc().getRDDStorageInfo()

# Via REST API
GET http://localhost:4040/api/v1/applications/{app}/executors
# Look for: totalGCTime, peakMemoryMetrics.JVMHeapMemory
```
